## 1. Setup & Imports

In [2]:
import pandas as pd #import pandas
import numpy as np #import numpy
import matplotlib.pyplot as plt #import matplotlib

## 2. Load Data

In [3]:
data = pd.read_csv("../data/bank_transactions_data_2.csv") #read the data from csv file

## 3. First Exploration

In [4]:
data.info() #get the information about the data
print(data.describe()) #get the statistical summary of the data
data.isnull().sum() #check for missing values

<class 'pandas.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionID            2512 non-null   str    
 1   AccountID                2512 non-null   str    
 2   TransactionAmount        2512 non-null   float64
 3   TransactionDate          2512 non-null   str    
 4   TransactionType          2512 non-null   str    
 5   Location                 2512 non-null   str    
 6   DeviceID                 2512 non-null   str    
 7   IP Address               2512 non-null   str    
 8   MerchantID               2512 non-null   str    
 9   Channel                  2512 non-null   str    
 10  CustomerAge              2512 non-null   int64  
 11  CustomerOccupation       2512 non-null   str    
 12  TransactionDuration      2512 non-null   int64  
 13  LoginAttempts            2512 non-null   int64  
 14  AccountBalance           2512 non-n

TransactionID              0
AccountID                  0
TransactionAmount          0
TransactionDate            0
TransactionType            0
Location                   0
DeviceID                   0
IP Address                 0
MerchantID                 0
Channel                    0
CustomerAge                0
CustomerOccupation         0
TransactionDuration        0
LoginAttempts              0
AccountBalance             0
PreviousTransactionDate    0
dtype: int64

## 4. Convert Datetime format

In [5]:
data["TransactionDate_dt"] = pd.to_datetime(data["TransactionDate"]) # needed as datetime to calculate time differences later
data["PreviousTransactionDate_dt"] = pd.to_datetime(data["PreviousTransactionDate"]) # needed as datetime to calculate time differences later

## 5. Account ID Analysis

In [6]:
print("Unique Account IDs:", data['AccountID'].nunique()) #explore how many unique account IDs are present in the dataset
print("Total Rows:", len(data))

Unique Account IDs: 495
Total Rows: 2512


## 6. Feature Engineering (Exploration)

In [ ]:
data_transaction = data.sort_values(by=['AccountID', 'TransactionDate_dt'])
data_transaction = data_transaction.reset_index(drop=True) 
data_transaction["PrevTransactionDate_calc"] = data_transaction.groupby("AccountID")["TransactionDate_dt"].shift(1) #calculate the previous transaction date for each account   

data_transaction["Transaction_diff"] = ((data_transaction["TransactionDate_dt"]-data_transaction["PrevTransactionDate_calc"])/pd.Timedelta(hours=1)).round(2)
data_transaction[['AccountID', 'TransactionDate_dt', 'PrevTransactionDate_calc', 'Transaction_diff']].head(15)

data["HomeLocation"] = data.groupby("AccountID")["Location"].transform(lambda x: x.mode()[0]) 
data["is_unusal_Location"] = (data["Location"] != data["HomeLocation"]).astype(int)
print(data[["HomeLocation", "Location","is_unusal_Location"]].head(15))

data["is_unusal_Location"].mean()
data.groupby("AccountID")["Location"].nunique().describe() #unique location are not suitable for fraud detection as some accounts have multiple locations, so we will try IP address instead

data.groupby("AccountID")["IP Address"].nunique().describe() #the same problem occurs with IP addresses 


   HomeLocation       Location  is_unusal_Location
0        Austin      San Diego                   1
1     Baltimore        Houston                   1
2    Louisville           Mesa                   1
3     Charlotte        Raleigh                   1
4       Atlanta        Atlanta                   0
5   Los Angeles  Oklahoma City                   1
6       Seattle        Seattle                   0
7        Dallas   Indianapolis                   1
8       Detroit        Detroit                   0
9       Chicago      Nashville                   1
10  Albuquerque    Albuquerque                   0
11      Phoenix        Memphis                   1
12     Columbus           Mesa                   1
13  Albuquerque        Memphis                   1
14       Boston     Louisville                   1


count    495.000000
mean       5.048485
std        2.273254
min        1.000000
25%        3.000000
50%        5.000000
75%        7.000000
max       12.000000
Name: IP Address, dtype: float64

In [18]:
columns_to_check = ["Location", "IP Address", "DeviceID", "MerchantID", "Channel", "TransactionType"]

results = {}
for col in columns_to_check:
    avg_unique = data.groupby("AccountID")[columns_to_check].nunique().mean()
    results[col] = avg_unique

# in ein DataFrame umwandeln zur übersichtlichen Darstellung
summary = pd.DataFrame(list(results.items()), columns=["Column", "AvgUniquePerAccount"])
print(summary)

            Column                                AvgUniquePerAccount
0         Location  Location           4.783838
IP Address        ...
1       IP Address  Location           4.783838
IP Address        ...
2         DeviceID  Location           4.783838
IP Address        ...
3       MerchantID  Location           4.783838
IP Address        ...
4          Channel  Location           4.783838
IP Address        ...
5  TransactionType  Location           4.783838
IP Address        ...
